In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Загружаем данные, сохраненные на предыдущем этапе
df = pd.read_csv('Churn_Modelling.csv')

print("Исходный размер датасета:", df.shape)
display(df.head(3))

Исходный размер датасета: (10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1


In [7]:
# 1. Удаление неинформативных признаков
# RowNumber (номер строки), CustomerId (ID клиента) и Surname (Фамилия) 
# никак не влияют на то, уйдет клиент или нет. Удаляем их, чтобы не путать модель.
df_cleaned = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

# 2. One-Hot Encoding (OHE) для категориальных признаков
# У нас есть колонки Geography (Страна) и Gender (Пол).
# Используем drop_first=True, чтобы избежать мультиколлинеарности (dummy variable trap).
# Например, если Gender_Male = 0, то очевидно, что это женщина. Второй столбец не нужен.
df_encoded = pd.get_dummies(df_cleaned, columns=['Geography', 'Gender'], drop_first=True)

print("Размер датасета после OHE:", df_encoded.shape)
display(df_encoded.head(3))

Размер датасета после OHE: (10000, 12)


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False


In [8]:
# Выделяем целевую переменную (y) и признаки (X)
X = df_encoded.drop(columns=['Exited'])
y = df_encoded['Exited']

# Разбиваем данные на обучающую (train) и тестовую (test) выборки (80% на 20%)
# ВАЖНО: параметр stratify=y сохраняет исходную пропорцию классов (80/20) в обеих выборках!
# Это наш первый шаг в борьбе с дисбалансом классов, обнаруженным на этапе EDA.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Размер обучающей выборки (X_train): {X_train.shape}")
print(f"Размер тестовой выборки (X_test): {X_test.shape}")

Размер обучающей выборки (X_train): (8000, 11)
Размер тестовой выборки (X_test): (2000, 11)


In [9]:
# Выбираем только числовые колонки (бинарные 0/1 масштабировать не обязательно)
numeric_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

# Инициализируем скалер
scaler = StandardScaler()

# ВАЖНО ДЛЯ КУРСОВОЙ (защита от Data Leakage - утечки данных):
# Метод .fit() вычисляет среднее и отклонение. Его мы применяем ТОЛЬКО к X_train!
# Если применить .fit() ко всему датасету до разбиения, модель "подглядит" в тестовые данные.
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])

# Для тестовой выборки используем только .transform(), опираясь на параметры из train
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

print("Масштабирование завершено. Первые строки X_train:")
display(X_train.head(3))

Масштабирование завершено. Первые строки X_train:


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
2151,1.058568,1.715086,0.684723,-1.226059,-0.910256,1,0,1.042084,False,False,True
8392,0.913626,-0.659935,-0.696202,0.413288,-0.910256,1,0,-0.623556,True,False,True
5006,1.079274,-0.184931,-1.731895,0.601687,0.808830,1,1,0.308128,True,False,False


In [10]:
# Сохраняем обработанные выборки в CSV файлы, 
# чтобы обучать модели в следующих ноутбуках без повторного запуска предобработки.
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Файлы X_train.csv, X_test.csv, y_train.csv и y_test.csv успешно сохранены!")

Файлы X_train.csv, X_test.csv, y_train.csv и y_test.csv успешно сохранены!
